In [25]:
# Import libraries
import pandas as pd
import numpy as np
import re

In [26]:
# Load enriched dataset
books_df = pd.read_csv(
    "../data/processed/books_enriched.csv"
)
books_df.head()

,title,price,availability,rating,product_url,category,description,upc,tax,number_available
0,A Light in the Attic,Â51.77,In stock,Three,http://books.toscrape.com/catalogue/a-light-in...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,Â£0.00,In stock (22 available)
1,Tipping the Velvet,Â53.74,In stock,One,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,Â£0.00,In stock (20 available)
2,Soumission,Â50.10,In stock,One,http://books.toscrape.com/catalogue/soumission...,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760,Â£0.00,In stock (20 available)
3,Sharp Objects,Â47.82,In stock,Four,http://books.toscrape.com/catalogue/sharp-obje...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,Â£0.00,In stock (20 available)
4,Sapiens: A Brief History of Humankind,Â54.23,In stock,Five,http://books.toscrape.com/catalogue/sapiens-a-...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,Â£0.00,In stock (20 available)


In [27]:
# Inspect dataset
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   title             1000 non-null   object
 1   price             1000 non-null   object
 2   availability      1000 non-null   object
 3   rating            1000 non-null   object
 4   product_url       1000 non-null   object
 5   category          992 non-null    object
 6   description       990 non-null    object
 7   upc               992 non-null    object
 8   tax               992 non-null    object
 9   number_available  992 non-null    object
dtypes: object(10)
memory usage: 78.2+ KB


In [28]:
# Check missing values
books_df.isnull().sum()

title                0
price                0
availability         0
rating               0
product_url          0
category             8
description         10
upc                  8
tax                  8
number_available     8
dtype: int64

In [29]:
# Clean price column
books_df["price"] = (
    books_df["price"]
    .str.replace(r"[^\d.]", "", regex=True)
)

books_df["price"] = pd.to_numeric(
    books_df["price"],
    errors="coerce"
)

In [30]:
# Clean tax column
books_df["tax"] = (
    books_df["tax"]
    .str.replace(r"[^\d.]", "", regex=True)
)

books_df["tax"] = pd.to_numeric(
    books_df["tax"],
    errors="coerce"
)

In [31]:
# Verify
books_df[["price", "tax"]].head()

,price,tax
0,51.77,0.0
1,53.74,0.0
2,50.10,0.0
3,47.82,0.0
4,54.23,0.0


In [32]:
# Convert ratings to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

books_df["rating"] = books_df["rating"].map(rating_map)
books_df.head()

,title,price,availability,rating,product_url,category,description,upc,tax,number_available
0,A Light in the Attic,51.77,In stock,3,http://books.toscrape.com/catalogue/a-light-in...,Poetry,It's hard to imagine a world without A Light i...,a897fe39b1053632,0.0,In stock (22 available)
1,Tipping the Velvet,53.74,In stock,1,http://books.toscrape.com/catalogue/tipping-th...,Historical Fiction,"""Erotic and absorbing...Written with starling ...",90fa61229261140a,0.0,In stock (20 available)
2,Soumission,50.10,In stock,1,http://books.toscrape.com/catalogue/soumission...,Fiction,"Dans une France assez proche de la nÃ´tre, un ...",6957f44c3847a760,0.0,In stock (20 available)
3,Sharp Objects,47.82,In stock,4,http://books.toscrape.com/catalogue/sharp-obje...,Mystery,"WICKED above her hipbone, GIRL across her hear...",e00eb4fd7b871a48,0.0,In stock (20 available)
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,http://books.toscrape.com/catalogue/sapiens-a-...,History,From a renowned historian comes a groundbreaki...,4165285e1663650f,0.0,In stock (20 available)


In [33]:
# Extract stock quantity
books_df["stock_quantity"] = (
    books_df["number_available"]
    .str.extract(r"(\d+)")
)

books_df["stock_quantity"] = pd.to_numeric(
    books_df["stock_quantity"],
    errors="coerce"
)

In [34]:
# Create stock status
books_df["stock_status"] = np.where(
    books_df["stock_quantity"] > 0,
    "In Stock",
    "Out of Stock"
)

In [35]:
# Create price category
books_df["price_category"] = pd.cut(
    books_df["price"],
    bins=[0,20,40,60],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [40]:
# Verify
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   title             1000 non-null   object  
 1   price             1000 non-null   float64 
 2   availability      1000 non-null   object  
 3   rating            1000 non-null   int64   
 4   product_url       1000 non-null   object  
 5   category          992 non-null    object  
 6   description       990 non-null    object  
 7   upc               992 non-null    object  
 8   tax               992 non-null    float64 
 9   number_available  992 non-null    object  
 10  stock_quantity    992 non-null    float64 
 11  stock_status      1000 non-null   object  
 12  price_category    1000 non-null   category
dtypes: category(1), float64(3), int64(1), object(8)
memory usage: 95.0+ KB


In [41]:
# Save cleaned dataset
books_df.to_csv(
    "../data/processed/books_cleaned.csv",
    index=False
)

The dataset was cleaned and transformed while preserving the original category structure provided by the source website. Categories such as "Default" and "Add a comment" are part of the website taxonomy and were retained for completeness.